
---
# RL01: Introduction to Reinforcement Learning
---

In this tutorial, we will inspect from a practical perspective the foundational algorithms of **tabular Reinforcement Learning** (RL), where the **environment** admits a **finite** number of **states** and the **agent** can choose among a **finite** set of possible **actions** in each of the states.

<center>
<img src="imgs/minecraft_interaction_loop.png" width=80% />

[*Original Image Source*](https://medium.com/student-technical-community-vit-vellore/a-brief-introduction-to-reinforcement-learning-6a74f5a61834)
</center>


The tutorial is structured as follows:
* In the **🧩 Problem Formulation** section we will learn how to represent _through code_ the elements that constitute a RL problem, i.e., the **environment** and the **policy**, and how to implement the interaction between the two.
* In the **📈 Prediction** section we will tackle the algorithms that allow to **evaluate** the **performance** of an agent in an environment and to estimate quantities of interest for the _control_ phase.
* Finally, in the **🎮 Control** section we will present the learning algorithms that allow to **optimize** the **behavior** of the agent, with the objective of learning the optimal policy for a given environment.

In this notebook you will find some **⭐ Exercise**s where you need to implement missing parts in the code. When you need to complete some code, the section is marked as:
```
# Your code goes here: -------------------------------------
# [... explanation of what you're required to implement ...]
# ----------------------------------------------------------
```

# ⚙️ Setup

Run the cell below to install and import all the notebook requirements.

In [1]:
import sys
if 'google.colab' in sys.modules:
    !pip install gymnasium
    !pip install numpy
    !pip install matplotlib
    !git clone https://github.com/gianmtedeschi/tutorial-rlss26.git

    %cd tutorial-rlss26/RL-basics

from utils import *
setup()

**Let's begin!**

# 🧩 Problem Formulation

## Environment

**Markov Decision Processes.** The fundamental model behind a RL problem is the **Markov Decision Process** (MDP), defined by:
* A finite set of **states**: $\mathcal{S}$, $|\mathcal{S}| < +\infty$.
* A subset of **terminal states**: $\mathcal{S}^- \subseteq \mathcal{S}$.
* A finite set of **actions**: $\mathcal{A}$, $|\mathcal{A}| < +\infty$.
* A state **transition** probability **matrix**: $P(\cdot \mid s, a) \in \Delta(\mathcal{S})$.
* A **reward function**: $R(s, a)$.
* An **initial state distribution**: $\mu_0 \in \Delta(\mathcal{S})$.
* A **horizon**: $H \in \mathbb{N}$.

_Remark._ The transition matrix and the initial state distribution are unknown: this is where learning comes into play!

**Gymnasium.** The `gymnasium` Python library models an MDP through the `Env` (_environment_) interface.

```python
class Env:
    def step(action: ActType) -> tuple[ObsType, float, bool, bool, dict]:
        ...
    
    def reset() -> tuple[ObsType, dict]:
        ...
```

The semantics of the interface are the following:
* The environment keeps track of the **current state** with an **internal attribute** `state` of type `ObsType`.
* The `step` method allows us to implement the **transition matrix** $P(\cdot \mid s, a)$ and the **reward function** $R(s, a)$. In particular, it takes as input an `action` of type `ActType` and returns a tuple containing:
    * The **next state** `next_state` of type `ObsType` and the **reward** `reward` of type `float`, generated according to the internal state `state` and the provided `action`.
    * A boolean flag `terminated` which evaluates to `True` if the `next_state` belongs to the subset of **terminal states** $\mathcal{S}^-$.
    * A boolean flag `truncated` which evaluates to `True` if the number of steps taken in the current episode reaches the **horizon** $H$.
    * A dictionary `info` containing auxiliary diagnostic data (which we will simply return as an empty dictionary `{}`).
* The `reset` method allows us to implement the **initial state distribution** $\mu_0$. In particular, it resets the internal state of the environment and returns the new initial state `initial_state` of type `ObsType`, alongside an initial `info` dictionary (again, we simply return `{}`).

_Remark._ The `Env` interface allows modeling the more general _Partially Observable_ MDP (POMDP), where the information provided to the agent is an element of the set of _observations_ $\mathcal{O}$ which, in general, can differ from the set of states $\mathcal{S}$. The class `ObsType` represents an element of the set $\mathcal{O}$. In this notebook, we will consider just the standard MDP case, where $\mathcal{O} = \mathcal{S}$. Thus `ObsType` is the class which represents the states of the MDP.

The `gymnasium` library provides a set of ready to use environment well-known in the RL literature that can be instantiated directly through the name of the desired environment.

```python
env = gym.make('<env-name>', render_mode='<mode>')
```

**📝 Example.** The _cliff walking_ environment models a grid world in which the agent has to learn to reach a goal position from a starting state, avoiding falling off the cliff.

<center>
<div style="position: relative; width: 900px; height: 300px; overflow: hidden; border: 1px solid #ccc;">
  <img src="https://gymnasium.farama.org/_images/cliff_walking.gif" style="position: absolute; top: 0; left: 0; width: 100%; height: 100%; object-fit: cover;" />
  <img src="imgs/cliff_walking_coords.svg" style="position: absolute; top: 0; left: 0; width: 100%; height: 100%; opacity: 0.9;" />
</div>
</center>

The environment models the following MDP:
* The **state** space is the set of pairs $\mathcal{S} = \{ 0, \dots, 3 \} \times \{ 0, \dots, 11 \}$, representing the current position of the agent in the $4 \times 12$ grid.
* The **terminal states** $\mathcal{S}^- \subset \mathcal{S}$ consist of the goal state $(3, 11)$ and the cliff states $\{ (3, c) \text{ s.t. } c \in \{1, \dots, 10\} \}$. Stepping into any of these states ends the episode naturally (evaluating `terminated` to `True`).
* The agent can do one of the following **actions**: $\mathcal{A} = \{$ Up, Right, Down, Left $\}$ represented with the numbers from $0$ to $3$ in the given order.
* The **state transition** is deterministic: the agent moves in the direction given by the action, unless there is a wall (grid boundary), in which case the state won't change.
* The **reward** function is shaped to encourage the agent to reach the goal as fast as possible without falling off the cliff. The agent receives a $-1$ reward for standard steps. If it steps onto the cliff, it receives a $-100$ reward.
* The agent **starts** from a **random state** safely above the cliff.
* The **horizon** is set to $H = 50$. If the agent survives for 50 steps of interaction without reaching a terminal state, the episode is artificially halted (evaluating `truncated` to `True`).

Below, you can find an **implementation** of the environment.

```python
UP    = 0
RIGHT = 1
DOWN  = 2
LEFT  = 3

class CliffWalkingEnv(Env):
    ROWS = 4
    COLS = 12
    CLIFF = {(3, c) for c in range(1, 11)}
    GOAL = (3, 11)

    def reset(self, *, seed=None, options=None):
        valid_states = [
            (r, c)
            for r in range(self.ROWS)
            for c in range(self.COLS)
            if (r, c) not in self.CLIFF and (r, c) != self.GOAL
        ]
        # Sample uniformly the initial state
        state = valid_states[np.random.randint(len(valid_states))]
        self.state = np.array(state, dtype=np.int32)
        # Count the number of interactions
        self.interactions = 0
        return self.state, {}

    def step(self, action: int):
        # If possible perform the action
        if action == UP and self.state[0] > 0:
            self.state = self.state + np.array([-1, 0], dtype=np.int32)
        elif action == RIGHT and self.state[1] < self.COLS - 1:
            self.state = self.state + np.array([0, 1], dtype=np.int32)
        elif action == DOWN and self.state[0] < self.ROWS - 1:
            self.state = self.state + np.array([1, 0], dtype=np.int32)
        elif action == LEFT and self.state[1] > 0:
            self.state = self.state + np.array([0, -1], dtype=np.int32)

        self.interactions += 1

        terminated = True if tuple(self.state) == self.GOAL else False
        truncated = self.interactions >= 500

        reward = -100 if tuple(self.state) in self.CLIFF else -1

        # When the agent falls off the cliff, terminate the episode
        if reward == -100:
           terminated = True

        return self.state, reward, terminated, truncated, {}
```

Let's **instantiate** the cliff environment.

In [2]:
CLIFF_WALKING_ENV_NAME = 'CliffWalking-RLSS-v0'
env = gym.make(CLIFF_WALKING_ENV_NAME, render_mode='rgb_array')

## Policy

A _policy_ describes the behavior of a RL agent inside of an environment.

**Mathematical Formulation.** Given an MDP with states $\mathcal{S}$ and actions $\mathcal{A}$, a (_Markovian_) policy $\pi$ is a map $\pi : \mathcal{S} \rightarrow \Delta(\mathcal{A})$.
<br> We say that the policy is deterministic whenever $\pi(s) = \delta_{a(s)}$ for every $s \in \mathcal{S}$, where $\delta_a$ is the Dirac Delta distribution over the action $a \in \mathcal{A}$.

**Implementation.** We implement policies through the following class.

In [3]:
class Policy(ABC):
    @abstractmethod
    def get_action(self, state):
        pass

The `get_action` method samples an action from the distribution induced by $\pi$ over the sate of actions $\mathcal{A}$, given `state`.

**⭐ Exercise.** Let's create a random policy for state spaces $\mathcal{S} \subset \mathbb{N}^d$ (i.e., grid worlds) and action spaces $\mathcal{A} = \{ 0, \dots, N_{\mathcal{A}} - 1 \}$ where $N_{\mathcal{A}}$ is the _number of actions_. Formally, the random policy is defined as $\pi(s) = \text{Unif}(\{ 0, \dots, N_{\mathcal{A}} - 1 \})$ for every $s \in \mathcal{S}$.

In [4]:
class RandomPolicy(Policy):
    def __init__(self, actions_cardinality):
        self.actions_cardinality = actions_cardinality
    
    def get_action(self, state: np.ndarray):
        # Solution: ------------------------------------------------
        return np.random.randint(0, self.actions_cardinality)
        # ----------------------------------------------------------

**⭐ Exercise.** Let's create an heuristic policy to solve the _cliff walking_ environment. <br>_Hint_: you can use the constants for the action mapping `UP = 0`, `RIGHT = 1`, etc., which are already defined.

In [5]:
class CliffWalkingHeuristicPolicy(Policy):
    def get_action(self, state: np.ndarray):
        # Solution: ------------------------------------------------
        if state[0] < 3 and state[1] < 11:
            return RIGHT
        elif state[0] == 3 and state[1] == 0:
            return UP
        else:
            return DOWN
        # ----------------------------------------------------------

# Learning Objective

**Expected Discounted Cumulative Reward.** The **goal** of a RL is to maximize the _expected discounted cumulative reward_ over the horizon $H$, for a given discount factor $\gamma \in [0, 1]$, defined as:
$$
J_\gamma(H) = \mathbb{E}_{\mu_0, P, \pi} \left[\sum_{h=0}^{H-1} \gamma^h R(S_h, A_h) \right].
$$

_Remark._ We follow the convention $0^0 = 1$ when $\gamma = 0$.

**⭐ Exercise.** Let's _implement_ a general **interaction loop** to simulate a given policy on a given environment, computing an estimate of the expected cumulative reward.

In [6]:
def evaluate_policy(env: gym.Env, policy, n_runs: int = 1, discount_factor: float = 1.0):
    mean_cumulative_reward = 0.0
    sum_squared_diffs = 0.0

    frames = []

    for i in tqdm(range(n_runs), desc="Evaluating Policy"):
        state, _ = env.reset()
        done = False
        cumulative_reward = 0.0
        time = 0

        frames = []

        while not done:
            frames.append(env.render())

            # Solution -------------------------------------------------
            action = policy.get_action(state)
            next_state, reward, terminated, truncated, _ = env.step(action)
            cumulative_reward += (discount_factor ** time) * reward
            # ----------------------------------------------------------

            time += 1
            state = next_state
            done = terminated or truncated

        # Welford's algorithm to compute running variance 
        diff_from_old_mean = cumulative_reward - mean_cumulative_reward
        mean_cumulative_reward += diff_from_old_mean / (i + 1)
        
        diff_from_new_mean = cumulative_reward - mean_cumulative_reward
        sum_squared_diffs += diff_from_old_mean * diff_from_new_mean

    # Compute sample standard deviation (Bessel's correction)
    if n_runs > 1:
        std_cumulative_reward = (sum_squared_diffs / (n_runs - 1)) ** 0.5
    else:
        std_cumulative_reward = 0.0

    std_cumulative_reward = 1.96 * (std_cumulative_reward / np.sqrt(n_runs))

    print(f"Evaluation - Mean Return: {mean_cumulative_reward:.2f}, Std Dev: {std_cumulative_reward:.2f}")

    return mean_cumulative_reward, std_cumulative_reward, frames

In [13]:
policy = CliffWalkingHeuristicPolicy()

mean_cumulative_reward, std_cumulative_reward, frames = evaluate_policy(env, policy, 100, 1)

animate_frames(frames)

Evaluating Policy: 100%|██████████| 100/100 [00:00<00:00, 170.94it/s]


Evaluation - Mean Return: -6.85, Std Dev: 3.37


# 📈 Prediction

# 🎮 Control